# How often does a free optical satellite get a usable look at Cushing, Oklahoma?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anefelo/anefelo-notebooks/blob/main/notebooks/2026-10-31-cushing-blind-weeks/cushing_blind_weeks.ipynb)

**Question.** For each ISO week from 2019 to 2025, did at least one Sentinel-2 image see the Cushing tank farms at least 80% cloud-free, and how does that compare with the cloud-independent Sentinel-1 radar cadence? Weeks without a usable optical look are weeks in which free optical imagery says nothing about storage at the largest US crude hub.

Part of [Anefelo](https://anefelo.substack.com), a newsletter on what satellite data says about the physical economy and its prices.

**Runs in** about 5–10 minutes on a free Colab CPU (dominated by ~1,000–2,000 small cloud-mask reads). No accounts or keys needed.

**Data:** Sentinel-2 Level-2A scene classification and Sentinel-1 GRD metadata via Microsoft Planetary Computer STAC; tank locations from OpenStreetMap (AOI check only). Full provenance and licences in section 2.

## 0. Setup

In [ ]:
# Colab and GitHub Actions both run this cell. Lower bounds only; exact versions are printed at the end.
%pip install -q "pystac-client>=0.8" "planetary-computer>=1.0" "rasterio>=1.4" "pandas>=2.0" "matplotlib>=3.8" "requests>=2.31"

In [ ]:
import datetime as dt
import json
import os
import platform
import random
import sys
from importlib import metadata
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FIG_DIR, RES_DIR = Path("figures"), Path("results")
FIG_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})


def get_secret(name):
    """Colab: the Secrets panel (key icon). GitHub Actions: a repository secret of the same name."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name) or None


SUMMARY = {}


def record(key, value, unit=None, note=None):
    """Every number the post cites is recorded here and written to results/summary.json."""
    if isinstance(value, (np.floating, np.integer)):
        value = value.item()
    SUMMARY[key] = {"value": value, "unit": unit, "note": note}
    return value


def save_figure(fig, name, credit):
    """Saves figures/<name>.png with the data credit printed on the image."""
    fig.text(0.01, -0.01, credit, fontsize=8, color="0.35", ha="left", va="top")
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print("saved", path)
    return path

## 1. Parameters

Everything a reader might change lives here: the area, the dates, the thresholds. Dates are fixed so a later run gives the same numbers (unless the catalogue itself changes).

In [ ]:
NOTEBOOK_NAME = "cushing_blind_weeks.ipynb"
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
S2_COLLECTION, S1_COLLECTION = "sentinel-2-l2a", "sentinel-1-grd"

# Area of interest: (min_lon, min_lat, max_lon, max_lat), WGS84 degrees.
# The Cushing, Oklahoma tank farms, about 5.6 x 11.8 km: the 1st-99th percentile extent of OpenStreetMap
# storage-tank centres plus 0.005 deg, as section 3a computed it on 2026-10-31. Fixed here so a later run
# gives the same numbers; section 3a re-checks it against current OSM data and prints the share inside.
# An earlier 9 x 10 km box, (-96.80, 35.92, -96.70, 36.01), held 86.5% of those tanks and stopped short
# of the terminals north of town.
BBOX = (-96.772, 35.92, -96.71, 36.026)
USE_OSM_BBOX = False          # True: replace BBOX with the OSM-derived box from section 3a

# Period: the ISO weeks whose Thursday falls between these dates (ISO convention)
PERIOD_START, PERIOD_END = "2019-01-01", "2025-12-31"

CLEAR_THRESHOLD = 0.80        # a look is "usable" if >= 80% of AOI pixels are clear
NOT_CLEAR_SCL = [0, 1, 3, 8, 9, 10]  # no data, saturated/defective, cloud shadow, cloud medium, cloud high, thin cirrus
SCL_RES_M = 20                # 20 = native SCL; 60 = lighter fallback read from overviews
MAX_WORKERS = 16              # parallel COG window reads
S1_MODE = "IW"
TILE_CLOUD_USABLE_PCT = 20    # secondary comparison only: tile-level eo:cloud_cover <= this

INPUTS = {"stac_url": STAC_URL, "s2_collection": S2_COLLECTION, "s1_collection": S1_COLLECTION,
          "bbox": BBOX, "use_osm_bbox": USE_OSM_BBOX, "period": [PERIOD_START, PERIOD_END],
          "clear_threshold": CLEAR_THRESHOLD, "not_clear_scl_classes": NOT_CLEAR_SCL,
          "scl_res_m": SCL_RES_M, "s1_mode": S1_MODE, "tile_cloud_usable_pct": TILE_CLOUD_USABLE_PCT}

## 2. Data and provenance

| Dataset | Provider and access | Collection / version | Licence and credit | Citation |
|---|---|---|---|---|
| Sentinel-2 Level-2A, scene classification layer (SCL, 20 m) | Copernicus / ESA, via Microsoft Planetary Computer STAC, https://planetarycomputer.microsoft.com/api/stac/v1 (links signed with `planetary-computer`) | `sentinel-2-l2a` | Free and open Copernicus data; "Contains modified Copernicus Sentinel data [years]" | ESA Sentinel-2 L2A product specification (SCL classes) |
| Sentinel-1 GRD, IW mode (metadata only: acquisition times) | Copernicus / ESA, via Microsoft Planetary Computer STAC | `sentinel-1-grd` | Free and open Copernicus data; "Contains modified Copernicus Sentinel data [years]" | — |
| Storage tanks (`man_made=storage_tank`), AOI check only | OpenStreetMap via Overpass API, https://overpass-api.de/api/interpreter | live query | ODbL; "© OpenStreetMap contributors" | — |

Context (not used as data): Mukherjee, Panayotov and Shon (2021), "Eye in the sky: Private satellites and government macro data", *Journal of Financial Economics*.

Accessed on the date printed in the last cell.

## 3. Method

1. **Check the box.** Pull OpenStreetMap storage tanks around Cushing and compare their extent with `BBOX` (printed and recorded; the box itself is fixed in the parameters).
2. **Find every optical look.** Search Sentinel-2 L2A for all scenes touching the box in the period, with no cloud filter. Where two tiles cover the same overpass, both are kept; a week counts if any of them is usable. Where the catalogue lists the same tile and overpass twice (a reprocessed copy), only the newest processing baseline is kept.
3. **Measure clear sky over the box, not the tile.** For each scene, read only the box from the SCL band and compute the share of pixels that are not no-data, cloud shadow, medium or high cloud, or thin cirrus. A scene with ≥ 80% clear is a usable look.
4. **Roll up to weeks.** A week (ISO, Monday to Sunday, UTC) is usable if it has at least one usable look. Report the share overall, by calendar month (month of the week's Thursday), and the longest run of consecutive blind weeks within each year.
5. **Radar contrast.** Count distinct Sentinel-1 IW GRD acquisitions (unique absolute orbits) touching the box each week. Radar sees through cloud, so its count is a cadence ceiling, not a clear-sky measure. No backscatter is analysed.
6. **Secondary check.** Compare the AOI-level result with the coarser tile-level `eo:cloud_cover ≤ 20%` rule.

### 3a. Check the box against OpenStreetMap storage tanks

This cell asks Overpass for storage tanks in a wide window around Cushing and prints the extent of their central 98% next to `BBOX`; if Overpass is down it says so and carries on.

In [ ]:
import requests

OSM_SEARCH = (-96.90, 35.85, -96.60, 36.10)  # wide window around Cushing
osm_bbox = None
try:
    s, w, n, e = OSM_SEARCH[1], OSM_SEARCH[0], OSM_SEARCH[3], OSM_SEARCH[2]
    q = f'[out:json][timeout:60];(way["man_made"="storage_tank"]({s},{w},{n},{e});node["man_made"="storage_tank"]({s},{w},{n},{e}););out center;'
    r = requests.post("https://overpass-api.de/api/interpreter", data={"data": q}, timeout=90,
                      headers={"User-Agent": "anefelo-notebooks (https://github.com/anefelo/anefelo-notebooks)"})
    r.raise_for_status()
    pts = []
    for el in r.json().get("elements", []):
        c = el.get("center", el)
        if "lat" in c and "lon" in c:
            pts.append((c["lon"], c["lat"]))
    pts = np.array(pts)
    print(f"OSM storage tanks in search window: {len(pts)}")
    if len(pts) >= 20:
        lo = np.percentile(pts, 1, axis=0)
        hi = np.percentile(pts, 99, axis=0)
        osm_bbox = tuple(round(float(v), 3) for v in (lo[0] - 0.005, lo[1] - 0.005, hi[0] + 0.005, hi[1] + 0.005))
        inside = ((pts[:, 0] >= BBOX[0]) & (pts[:, 0] <= BBOX[2]) & (pts[:, 1] >= BBOX[1]) & (pts[:, 1] <= BBOX[3])).mean()
        print("OSM-derived box (1st-99th percentile of tank centres, +0.005 deg):", osm_bbox)
        print("Parameter BBOX:                                              ", BBOX)
        print(f"Share of OSM tanks inside BBOX: {inside:.1%}")
        record("osm_tank_count", int(len(pts)), "tanks", f"OSM man_made=storage_tank within {OSM_SEARCH}")
        record("osm_tanks_inside_bbox_share", round(float(inside), 4), "share", "OSM tank centres falling inside BBOX")
        record("osm_derived_bbox", list(osm_bbox), "degrees (min_lon, min_lat, max_lon, max_lat)", "1st-99th percentile of tank centres plus 0.005 deg; for checking BBOX")
except Exception as exc:
    print("Overpass check skipped:", type(exc).__name__, str(exc)[:200])

if USE_OSM_BBOX and osm_bbox:
    BBOX = osm_bbox
    INPUTS["bbox"] = BBOX
print("AOI used:", BBOX)
record("aoi_bbox", list(BBOX), "degrees (min_lon, min_lat, max_lon, max_lat)", "WGS84; OSM storage-tank extent (section 3a)")

### 3b. The ISO-week calendar and the analysis helpers

This cell defines the weekly calendar for the period and the small, pure functions that turn per-scene clear fractions into weekly results; they are tested on a synthetic example right after.

In [ ]:
import pandas as pd


def week_calendar(start, end):
    """All ISO weeks whose Thursday lies in [start, end]. Index: Monday of each week."""
    first_thu = pd.Timestamp(start) + pd.Timedelta(days=(3 - pd.Timestamp(start).weekday()) % 7)
    thursdays = pd.date_range(first_thu, pd.Timestamp(end), freq="7D")
    mondays = thursdays - pd.Timedelta(days=3)
    iso = thursdays.isocalendar()
    return pd.DataFrame({"iso_year": iso["year"].to_numpy(), "iso_week": iso["week"].to_numpy(),
                         "month": thursdays.month, "thursday": thursdays}, index=pd.Index(mondays, name="monday"))


def clear_fraction(scl, not_clear=NOT_CLEAR_SCL):
    """Share of AOI pixels that are clear: everything except no-data, shadow, cloud and cirrus classes."""
    scl = np.asarray(scl)
    if scl.size == 0:
        return np.nan
    return float((~np.isin(scl, not_clear)).mean())


def weekly_flags(acq, weeks, threshold=CLEAR_THRESHOLD):
    """acq: DataFrame with 'datetime' (UTC) and 'clear_frac'. Returns weeks with n_looks, n_usable, usable."""
    out = weeks.copy()
    if len(acq):
        a = acq.dropna(subset=["clear_frac"]).copy()
        dt_naive = pd.to_datetime(a["datetime"], utc=True, format="ISO8601").dt.tz_localize(None)
        a["monday"] = dt_naive.dt.normalize() - pd.to_timedelta(dt_naive.dt.day_of_week, unit="D")
        a["usable"] = a["clear_frac"] >= threshold
        g = a.groupby("monday").agg(n_looks=("usable", "size"), n_usable=("usable", "sum"))
        out = out.join(g)
    else:
        out["n_looks"], out["n_usable"] = 0, 0
    out[["n_looks", "n_usable"]] = out[["n_looks", "n_usable"]].fillna(0).astype(int)
    out["usable"] = out["n_usable"] > 0
    return out


def longest_run(flags):
    """Longest run of consecutive True values in a boolean sequence."""
    best = cur = 0
    for f in flags:
        cur = cur + 1 if f else 0
        best = max(best, cur)
    return best


# Synthetic test: 3 weeks, week 2 has only a cloudy look, week 3 has none
_w = week_calendar("2024-01-01", "2024-01-21")
_acq = pd.DataFrame({"datetime": ["2024-01-02T17:20:00Z", "2024-01-04T17:20:00Z", "2024-01-10T17:20:00Z"],
                     "clear_frac": [0.5, clear_fraction(np.array([[4, 5], [6, 7]])), clear_fraction(np.array([[9, 9], [4, 8]]))]})
_f = weekly_flags(_acq, _w)
assert list(_f["usable"]) == [True, False, False] and list(_f["n_looks"]) == [2, 1, 0], _f
assert longest_run(~_f["usable"]) == 2 and longest_run([]) == 0
assert clear_fraction(np.array([0, 1, 3, 8, 9, 10, 4, 5, 6, 11])) == 0.4
weeks = week_calendar(PERIOD_START, PERIOD_END)
print(f"{len(weeks)} ISO weeks from {weeks['iso_year'].iloc[0]}-W{weeks['iso_week'].iloc[0]:02d} "
      f"to {weeks['iso_year'].iloc[-1]}-W{weeks['iso_week'].iloc[-1]:02d}; helpers pass the synthetic test")

### 3c. Find every Sentinel-2 and Sentinel-1 acquisition over the box

This cell searches the Planetary Computer catalogue year by year (search window padded to whole ISO weeks) and stops if either search comes back empty.

In [ ]:
import planetary_computer
import pystac_client

catalog = pystac_client.Client.open(STAC_URL, modifier=planetary_computer.sign_inplace)
search_start = weeks.index[0].date()
search_end = (weeks.index[-1] + pd.Timedelta(days=6)).date()


def search_all(collection, query=None):
    items = []
    for y in range(search_start.year, search_end.year + 1):
        a = max(search_start, dt.date(y, 1, 1))
        b = min(search_end, dt.date(y, 12, 31))
        if a > b:
            continue
        s = catalog.search(collections=[collection], bbox=BBOX, datetime=f"{a}T00:00:00Z/{b}T23:59:59Z",
                           query=query, limit=500)
        got = list(s.items())
        print(f"  {collection} {y}: {len(got)} items")
        items += got
    return items


s2_items = search_all(S2_COLLECTION)
s1_items = search_all(S1_COLLECTION, query={"sar:instrument_mode": {"eq": S1_MODE}})
assert s2_items, "No Sentinel-2 L2A items found over the AOI"
assert s1_items, "No Sentinel-1 GRD IW items found over the AOI"
assert all("SCL" in it.assets for it in s2_items[:50]), "SCL asset missing from Sentinel-2 items"
print(f"Sentinel-2 scenes: {len(s2_items)} | Sentinel-1 GRD IW slices: {len(s1_items)}")

### 3d. Read the cloud mask over the box for every Sentinel-2 scene

This cell reads only the AOI window of each scene's SCL band (in parallel, with retries) and computes the clear fraction; at `SCL_RES_M = 60` it reads a 3x-decimated window from the overviews instead.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

import rasterio
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds

GDAL_ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.TIF",
                GDAL_HTTP_MAX_RETRY="3", GDAL_HTTP_RETRY_DELAY="2", VSI_CACHE="TRUE")


def read_scl_window(href, bbox, res_m=SCL_RES_M):
    with rasterio.Env(**GDAL_ENV), rasterio.open(href) as src:
        bounds = transform_bounds("EPSG:4326", src.crs, *bbox, densify_pts=21)
        win = from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
        factor = max(1, int(round(res_m / abs(src.res[0]))))
        shape = (max(1, int(win.height) // factor), max(1, int(win.width) // factor))
        # boundless: pixels outside the tile come back as 0 (no data) and so count as not clear
        return src.read(1, window=win, out_shape=shape, resampling=Resampling.nearest, boundless=True, fill_value=0)


def scene_clear_fraction(item, tries=3):
    for k in range(tries):
        try:
            href = item.assets["SCL"].href if k == 0 else planetary_computer.sign(item).assets["SCL"].href
            scl = read_scl_window(href, BBOX)
            return clear_fraction(scl), int(scl.size), None
        except Exception as exc:
            err = f"{type(exc).__name__}: {str(exc)[:120]}"
            time.sleep(2 * (k + 1))
    return np.nan, 0, err


t0 = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    res = list(pool.map(scene_clear_fraction, s2_items))
print(f"read {len(s2_items)} SCL windows in {time.time() - t0:.0f} s")

s2 = pd.DataFrame({
    "id": [it.id for it in s2_items],
    "datetime": [it.datetime.isoformat() for it in s2_items],
    "platform": [it.properties.get("platform") for it in s2_items],
    "tile_cloud_pct": [it.properties.get("eo:cloud_cover") for it in s2_items],
    "tile": [it.properties.get("s2:mgrs_tile") for it in s2_items],
    "baseline": [it.properties.get("s2:processing_baseline") for it in s2_items],
    "clear_frac": [r[0] for r in res], "n_pixels": [r[1] for r in res], "error": [r[2] for r in res],
}).sort_values("datetime").reset_index(drop=True)
n_fail = int(s2["clear_frac"].isna().sum())
print(f"failed reads: {n_fail}", s2.loc[s2["error"].notna(), "error"].head(3).tolist())
assert n_fail <= 0.02 * len(s2), f"{n_fail} of {len(s2)} SCL reads failed; rerun the cell"
print(s2.describe(include="all").T[["count", "mean"]].dropna(how="all"))

This cell checks whether the catalogue lists the same tile and overpass more than once (a reprocessed copy, for example), prints the yearly counts by tile and processing baseline, and keeps one item per tile and overpass: the newest processing baseline, then the latest processing date.

In [ ]:
s2["year"] = pd.to_datetime(s2["datetime"], utc=True, format="ISO8601").dt.year
s2["overpass"] = s2["datetime"].str[:16]  # sensing time to the minute: one overpass of one tile
dup = s2.duplicated(["tile", "overpass"], keep=False)
per_year = pd.DataFrame({
    "items": s2.groupby("year").size(),
    "distinct_tile_overpasses": s2.drop_duplicates(["tile", "overpass"]).groupby("year").size(),
    "items_sharing_tile_overpass": s2[dup].groupby("year").size(),
}).fillna(0).astype(int)
print(per_year, "\n")
print(pd.crosstab(s2["year"], s2["tile"].fillna("n/a")), "\n")
print(pd.crosstab(s2["year"], s2["baseline"].fillna("n/a")), "\n")
groups = s2[dup].groupby(["tile", "overpass"])["baseline"].nunique()
print(f"tile-overpass pairs listed more than once: {len(groups)} | of which with different baselines: {int((groups > 1).sum())}")
same = s2.set_index(["tile", "overpass"]).index.isin(groups[groups == 1].index)
print("pairs with the same baseline:")
print(s2.loc[same, ["id", "baseline", "clear_frac"]].to_string(), "\n")
record("s2_catalogue_items", len(s2), "items", "Sentinel-2 L2A catalogue items touching the AOI, before removing repeated tile-overpass items")
record("s2_items_sharing_tile_overpass", int(dup.sum()), "items", "catalogue items whose MGRS tile and overpass minute match another item (mostly reprocessed copies)")

# Keep one item per tile and overpass: newest processing baseline, then latest processing date (end of the item id)
s2 = (s2.assign(_b=s2["baseline"].fillna(""), _p=s2["id"].str[-15:])
        .sort_values(["_b", "_p"])
        .drop_duplicates(["tile", "overpass"], keep="last")
        .drop(columns=["_b", "_p"])
        .sort_values("datetime").reset_index(drop=True))
print(f"kept {len(s2)} items, one per tile and overpass")


### 3e. Roll up to weeks and count radar passes

This cell applies the 80% rule, aggregates to ISO weeks, and counts distinct Sentinel-1 acquisitions per week (one per absolute orbit, so adjacent slices of one pass count once).

In [ ]:
wk = weekly_flags(s2, weeks)

s1 = pd.DataFrame({"datetime": pd.to_datetime([it.datetime for it in s1_items], utc=True).tz_localize(None),
                   "platform": [it.properties.get("platform") for it in s1_items],
                   "orbit": [it.properties.get("sat:absolute_orbit") for it in s1_items]})
s1["orbit"] = s1["orbit"].fillna(s1["datetime"].dt.strftime("%Y%m%d%H"))  # fallback: one pass per hour
s1 = s1.sort_values("datetime").drop_duplicates(["platform", "orbit"])
s1["monday"] = s1["datetime"].dt.normalize() - pd.to_timedelta(s1["datetime"].dt.day_of_week, unit="D")
wk["s1_acq"] = s1.groupby("monday").size().reindex(wk.index).fillna(0).astype(int)

# Secondary, coarser rule: any scene in the week with tile-level eo:cloud_cover <= threshold
tile = s2.assign(clear_frac=(s2["tile_cloud_pct"] <= TILE_CLOUD_USABLE_PCT).astype(float))
wk["usable_tile_rule"] = weekly_flags(tile, weeks, threshold=0.5)["usable"]

years = sorted(wk["iso_year"].unique())
yr = wk.groupby("iso_year").agg(weeks=("usable", "size"), usable_weeks=("usable", "sum"),
                                 s2_looks=("n_looks", "sum"), s1_acq=("s1_acq", "sum"))
yr["usable_share"] = yr["usable_weeks"] / yr["weeks"]
yr["longest_blind_run"] = [longest_run(~wk.loc[wk["iso_year"] == y, "usable"].to_numpy()) for y in yr.index]
mo = wk.groupby("month").agg(weeks=("usable", "size"), usable_share=("usable", "mean"),
                             s1_mean_per_week=("s1_acq", "mean"), s1_any_share=("s1_acq", lambda x: (x > 0).mean()))
print(yr, "\n")
print(mo.round(3))

### 3f. Record every number

This cell passes each figure the post may cite through `record`, with units and notes.

In [ ]:
n_weeks = len(wk)
record("weeks_total", n_weeks, "ISO weeks", f"ISO weeks with Thursday in {PERIOD_START}..{PERIOD_END}")
record("s2_scenes", len(s2), "scenes", "Sentinel-2 L2A items touching the AOI, one per tile and overpass (tiles counted separately)")
record("s2_scl_read_failures", n_fail, "scenes", "excluded from the weekly roll-up")
record("s2_scene_usable_share", round(float((s2["clear_frac"] >= CLEAR_THRESHOLD).mean()), 4), "share of scenes",
       f"scenes with >= {CLEAR_THRESHOLD:.0%} of AOI pixels clear (SCL not in {NOT_CLEAR_SCL})")
record("share_weeks_usable", round(float(wk["usable"].mean()), 4), "share of weeks",
       "weeks with at least one Sentinel-2 acquisition >= 80% clear over the AOI")
record("weeks_usable", int(wk["usable"].sum()), "ISO weeks", None)
record("weeks_blind", int((~wk["usable"]).sum()), "ISO weeks", "no usable optical look (cloudy or no acquisition)")
record("weeks_no_s2_acquisition", int((wk["n_looks"] == 0).sum()), "ISO weeks", "no Sentinel-2 acquisition at all (cadence gap)")
record("longest_blind_run_overall", longest_run(~wk["usable"].to_numpy()), "consecutive ISO weeks", "across year boundaries")
for y, row in yr.iterrows():
    record(f"share_weeks_usable_{y}", round(float(row["usable_share"]), 4), "share of weeks", f"ISO year {y}")
    record(f"longest_blind_run_{y}", int(row["longest_blind_run"]), "consecutive ISO weeks", f"within ISO year {y} (runs cut at year boundary)")
    record(f"s2_looks_{y}", int(row["s2_looks"]), "scenes", f"Sentinel-2 scenes read in ISO year {y}; drops when only one satellite flew")
for m, row in mo.iterrows():
    mname = dt.date(2000, m, 1).strftime("%b").lower()
    record(f"share_weeks_usable_{mname}", round(float(row["usable_share"]), 4), "share of weeks",
           f"weeks whose Thursday falls in month {m}, all years ({int(row['weeks'])} weeks)")
record("s1_acq_per_week_mean", round(float(wk["s1_acq"].mean()), 3), "acquisitions per week",
       "distinct Sentinel-1 IW GRD passes (unique platform+absolute orbit) touching the AOI; counts only")
record("s1_share_weeks_with_acq", round(float((wk["s1_acq"] > 0).mean()), 4), "share of weeks",
       "weeks with >= 1 Sentinel-1 IW acquisition; radar is cloud-independent")
for y in years:
    record(f"s1_acq_per_week_mean_{y}", round(float(wk.loc[wk["iso_year"] == y, "s1_acq"].mean()), 3), "acquisitions per week", f"ISO year {y}")
record("share_weeks_usable_tile_rule", round(float(wk["usable_tile_rule"].mean()), 4), "share of weeks",
       f"secondary: weeks with a scene whose tile-level eo:cloud_cover <= {TILE_CLOUD_USABLE_PCT}%")
record("agreement_aoi_vs_tile_rule", round(float((wk["usable"] == wk["usable_tile_rule"]).mean()), 4), "share of weeks",
       "weeks where the AOI-level and tile-level rules agree")

## 4. Results

Each figure states its takeaway in the title and carries its data credit.

Figure 1 is a year-by-week calendar: dark weeks had a usable optical look, mid-tone weeks had only cloudy looks, light weeks had no Sentinel-2 acquisition at all.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

state = np.where(wk["usable"], 2, np.where(wk["n_looks"] > 0, 1, 0))
grid = np.full((len(years), 53), np.nan)
for (y, w), s in zip(zip(wk["iso_year"], wk["iso_week"]), state):
    grid[years.index(y), w - 1] = s
cmap = ListedColormap(["#f0e442", "#9e9ac8", "#2c1e6b"])  # no look / cloudy only / usable (colour-blind safe)
cmap.set_bad("white")
fig, ax = plt.subplots(figsize=(11, 4.2), constrained_layout=True)
ax.imshow(np.ma.masked_invalid(grid), cmap=cmap, norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5], 3), aspect="auto",
          interpolation="nearest", extent=(0.5, 53.5, len(years) - 0.5, -0.5))
ax.set_yticks(range(len(years)), [f"{y}  ({yr.loc[y, 'usable_share']:.0%})" for y in years])
ax.set_xticks([1, 5, 9, 13, 18, 22, 26, 31, 35, 39, 44, 48, 52])
ax.set_xlabel("ISO week of year")
ax.set_ylabel("ISO year (share of weeks usable)")
ax.set_xticks(np.arange(1.5, 53.5), minor=True)
ax.set_yticks(np.arange(0.5, len(years) - 0.5), minor=True)
ax.grid(which="minor", color="white", linewidth=0.6)
ax.tick_params(which="minor", length=0)
ax.legend(handles=[Patch(color="#2c1e6b", label="usable look (AOI >= 80% clear)"),
                   Patch(color="#9e9ac8", label="looks, all cloudy"),
                   Patch(color="#f0e442", label="no Sentinel-2 acquisition")],
          loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, frameon=False)
share = SUMMARY["share_weeks_usable"]["value"]
ax.set_title(f"Sentinel-2 got a usable look at the Cushing tank farms in {share:.0%} of weeks, {years[0]}-{years[-1]}", loc="left")
s2_years = pd.to_datetime(s2['datetime'], utc=True, format="ISO8601").dt.year
year_str = f"{s2_years.min()}-{s2_years.max()}"
save_figure(fig, "fig1_cushing_usable_weeks_calendar", credit=f"Contains modified Copernicus Sentinel data {year_str}")
plt.show()

Figure 2 shows the share of usable weeks by calendar month, with the mean number of Sentinel-1 radar passes per week on a second axis as the cloud-independent reference.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
labels = [dt.date(2000, m, 1).strftime("%b") for m in mo.index]
bars = ax.bar(labels, mo["usable_share"] * 100, color="#3b528b", width=0.7, label="Sentinel-2: weeks with a usable look")
ax.bar_label(bars, labels=[f"{v:.0%}" for v in mo["usable_share"]], fontsize=8, padding=2)
ax.axhline(share * 100, color="0.4", ls="--", lw=1)
ax.text(-0.45, share * 100 + 1.5, f"all months {share:.0%}", ha="left", fontsize=8, color="0.3")
ax.set_ylim(0, 105)
ax.set_ylabel("Share of ISO weeks with a usable optical look (%)")
ax2 = ax.twinx()
ax2.plot(labels, mo["s1_mean_per_week"], color="#e08214", marker="o", lw=1.5, label="Sentinel-1 IW passes per week (mean)")
ax2.set_ylim(0, max(3, float(mo["s1_mean_per_week"].max()) * 1.6))
ax2.set_ylabel("Sentinel-1 acquisitions per week (mean)")
ax2.spines["right"].set_visible(True)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left", frameon=False, fontsize=8)
worst = mo["usable_share"].idxmin()
ax.set_title(f"Optical looks at Cushing are scarcest in {dt.date(2000, worst, 1):%B} "
             f"({mo.loc[worst, 'usable_share']:.0%} of weeks); radar passes ignore cloud "
             f"({SUMMARY['s1_share_weeks_with_acq']['value']:.0%} of weeks with >= 1 pass)", loc="left", fontsize=10)
save_figure(fig, "fig2_cushing_usable_share_by_month", credit=f"Contains modified Copernicus Sentinel data {year_str}")
plt.show()

In [ ]:
summary = {
    "meta": {
        "notebook": NOTEBOOK_NAME,
        "generated_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "inputs": INPUTS,
    },
    "values": SUMMARY,
}
(RES_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))

## 5. What this does not show

- **Usable is not useful.** An 80%-clear 10–20 m Sentinel-2 look shows where tanks are, not how full they are. Inventory estimates rely on floating-roof shadows seen in sub-metre commercial imagery; this notebook measures only when a free optical sensor could see the site at all.
- **Cloud mask errors.** SCL misses some thin cloud and haze and sometimes flags bright tank roofs as cloud; the 80% threshold and class list are choices (see parameters). Shadow from cloud outside the box is counted, cloud just outside is not.
- **Box choice.** The share depends on the box: a larger box is harder to see 80% clear than the tanks alone. The box is the OpenStreetMap tank extent trimmed at the 1st and 99th percentiles, so a few outlying tanks fall outside it, and tanks missing from OSM are not counted.
- **Cadence changes over time.** Sentinel-2B (from late 2021 for some months) and Sentinel-1B (lost December 2021) outages, and the later arrival of Sentinel-2C and Sentinel-1C, change how many looks a week can have; per-year counts are recorded.
- **Weeks, not announcement days.** An ISO week is Monday to Sunday; the EIA reports Wednesday stocks as of the prior Friday, so a usable look in the same ISO week may not precede the announcement. No price data are analysed here.
- **Radar counts only.** Sentinel-1 acquisitions are counted, not interpreted; whether 10 m C-band backscatter tracks tank fill is a separate question.
- **One site.** Cushing is one hub on the southern Great Plains; other storage sites have different cloud climates.

In [ ]:
# Environment, so a reader can match versions if a number differs
packages = ["pystac-client", "pystac", "planetary-computer", "rasterio", "pandas", "numpy", "matplotlib", "requests"]
print("generated (UTC):", dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"))
print("python:", platform.python_version(), "| colab:", IN_COLAB)
for name in packages:
    try:
        print(f"{name}: {metadata.version(name)}")
    except metadata.PackageNotFoundError:
        print(f"{name}: not installed")